# पाठ 05 - एजेंटिक RAG


## सेटअप

यह नोटबुक Microsoft Agent Framework का उपयोग करके Agentic RAG (रिकवरी ऑगमेंटेड जनरेशन) पैटर्न को प्रदर्शित करता है।

**पूर्वापेक्षाएँ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — आपका Azure AI Search सेवा endpoint
- `AZURE_SEARCH_API_KEY` — आपका Azure AI Search API key
- पर्यावरण चर के माध्यम से कॉन्फ़िगर किया गया Azure OpenAI deployment
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजेंटिक RAG क्या है?

पारंपरिक RAG एक निश्चित पाइपलाइन का पालन करता है: दस्तावेज़ प्राप्त करें, फिर उत्तर जनरेट करें। **एजेंटिक RAG** इससे आगे बढ़ता है, एजेंट को यह निर्णय लेने की स्वायत्तता देता है कि **कब** और **कैसे** जानकारी प्राप्त करनी है।

एजेंटिक RAG के साथ, एजेंट कर सकता है:
- **निर्णय लें** कि उत्तर देने से पहले जानकारी पुनः प्राप्त करना आवश्यक है या नहीं
- **चुनें** कि कौन सा डेटा स्रोत या उपकरण क्वेरी करना है
- **मूल्यांकन करें** प्राप्त परिणामों का और यदि पहली कोशिश अपर्याप्त हो तो आगे की पुनः प्राप्ति करें
- **संयोजित करें** कई पुनः प्राप्ति चरणों से जानकारी को एक सुसंगत उत्तर में

यह एजेंट को एक स्थिर पुनः प्राप्त-फिर-जनरेट पाइपलाइन की तुलना में अधिक लचीला और सटीक बनाता है।


## खोज उपकरण बनाना

Agentic RAG में, बाहरी डेटा स्रोतों को **tools** के रूप में रैप किया जाता है जिन्हें एजेंट आवश्यकता अनुसार कॉल कर सकता है। इससे एजेंट के लिए रिट्रीवल को केवल एक अन्य क्रिया के रूप में लेना संभव होता है, न कि एक अनिवार्य कदम के रूप में।

नीचे हमने एक यात्रा ज्ञान आधार परिभाषित किया है और इसे एक उपकरण के रूप में प्रस्तुत किया है जिसे एजेंट गंतव्य जानकारी देखने के लिए कॉल कर सकता है।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेंट बनाना

अब हम एक एजेंट बनाते हैं जिसे निर्देश दिया गया है कि **सदा उत्तर देने से पहले जानकारी प्राप्त करें**। एजेंट अपने उत्तरों को ज्ञान आधार में आधारित करने के लिए `search_travel_knowledge` टूल का उपयोग करता है, बजाय अपने स्वयं के प्रशिक्षण डेटा पर भरोसा करने के।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्त पुनर्प्राप्ति — मेकर-चेककर पैटर्न

Agentic RAG का एक प्रमुख लाभ है **पुनरावृत्त पुनर्प्राप्ति**। एजेंट कई दौर के खोज कर सकता है ताकि वह अपनी प्रारंभिक खोजों को सत्यापित, परिष्कृत या विस्तार कर सके — बिलकुल एक "मेकर-चेककर" कार्यप्रवाह के समान:

1. **मेकर चरण**: एजेंट प्रारंभिक जानकारी पुनःप्राप्त करता है और एक उत्तर का प्रारूप तैयार करता है।
2. **चेककर चरण**: एजेंट विवरणों को सत्यापित करने या अन्तराल भरने के लिए अतिरिक्त पुनर्प्राप्तियाँ करता है।

नीचे, एजेंट से एक ऐसा प्रश्न पूछा गया है जिसमें कई गंतव्यों की तुलना करनी होती है, जिससे वह कई बार खोज करने के लिए प्रेरित होता है।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

इस पाठ में आपने सीखा कि Microsoft Agent Framework का उपयोग करके एक **Agentic RAG** सिस्टम कैसे बनाया जाए:

- **Agentic RAG** एजेंटों को स्वायत्त रूप से यह तय करने देता है कि सूचना कब पुनःप्राप्त करनी है, जिससे पुनःप्राप्ति स्थिर के बजाय गतिशील हो जाती है।
- **टूल्स को डेटा स्रोत के रूप में उपयोग करना**: बाहरी ज्ञान आधार (जैसे Azure AI Search) को टूल्स के रूप में आवृत किया जाता है जिन्हें एजेंट कॉल कर सकता है।
- **आवर्ती पुनःप्राप्ति**: मेकर-चेकर पैटर्न एजेंट को कई पुनःप्राप्ति दौर करने की अनुमति देता है — खोज करना, सत्यापित करना, और अंतिम उत्तर देने से पहले उसे परिष्कृत करना।

उत्पादन में, आप मेमोरी में मौजूद `TRAVEL_KNOWLEDGE_BASE` को बड़े पैमाने पर यात्रा दस्तावेज पुनःप्राप्ति को संभालने के लिए एक वास्तविक Azure AI Search इंडेक्स से बदलेंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
